# 모델 서비스하기 핵심 정리저장/불러오기, 이어서 학습, Flask/TFJS 서비스, 서버 안정화

## 1. 모델 저장/불러오기 — SavedModel 형식이 기본(구조+가중치+compile정보까지 다 저장)

In [ ]:
from tensorflow import kerasmodel.fit(x_train, y_train, epochs=10)   # 학습model.save("my_model")                    # SavedModel 형식으로 통째로 저장 (폴더 형태)loaded_model = keras.models.load_model("my_model")  # 다시 불러오기 -> 바로 predict/fit 가능

### 1-1. 이어서 학습하기 — `initial_epoch`으로 어디서부터 재개할지 지정

In [ ]:
# 예: 이미 20 epoch까지 학습된 모델을 40 epoch까지 이어서 학습loaded_model.fit(  x_train, y_train,  initial_epoch=20,  # 20 epoch까지 했다고 표시 (진행상황 표시용, 여기서부터 다시 도는 건 아님)  epochs=40            # 목표는 40 epoch까지)

### 1-2. 체크포인트에서 불러와서 이어 학습하기 — ModelCheckpoint로 저장해둔 지점부터 재개

In [ ]:
loaded_model = keras.models.load_model("checkpoints/cp-0020.ckpt")  # 20번째 epoch 체크포인트 로드loaded_model.fit(x_train, y_train, initial_epoch=20, epochs=40)       # 이어서 학습

## 2. 모델 서비스하기 — Flask로 서버에서 predict 응답해주기

In [ ]:
from flask import Flask, requestimport tensorflow as tfapp = Flask(__name__)@app.route('/predict', methods=['POST'])def predict():  inputdata = request.json['data']       # 요청에서 입력 데이터 꺼내기  res = model.predict([inputdata])        # 모델로 예측  return {"result": res.tolist()}          # 결과 반환 (numpy는 json 직렬화 안되니 tolist())if __name__ == '__main__':  model = tf.keras.models.load_model("my_model")  # 서버 켤 때 딱 한 번만 모델 로드 (요청마다 로드하면 X)  app.run(host='localhost', port=8080)

### 2-1. TensorFlow.js — 모델을 브라우저(JS)에서 바로 돌리고 싶을 때 변환

In [ ]:
import tensorflowjs as tfjstfjs.converters.save_keras_model(model, "tfjs_target_dir")  # 케라스 모델 -> TFJS용 포맷으로 변환/저장

JS 쪽 사용법 (파이썬 코드 아님, 참고용):

```jsimport * as tf from '@tensorflow/tfjs';// 변환된 모델 불러오기const model = await tf.loadLayersModel('https://foo.bar/tfjs_artifacts/model.json');const example = tf.fromPixels(webcamElement);  // 웹캠 등에서 입력 텐서 생성const prediction = model.predict(example);      // 브라우저에서 바로 예측```

## 3. 서버 안정화 — 딥러닝 모델은 자원을 많이 먹어서 요청을 무한정 받으면 서버가 죽음

### 3-1. 동시 작업 수 제한 — 처리 중인 작업 수를 세서 너무 많으면 요청을 거절

In [ ]:
max_works = 5     # 동시에 처리 가능한 최대 작업 수 (자원 상황 봐서 정함)num_works = 0      # 현재 처리 중인 작업 수 (전역 카운터)@app.route('/predict', methods=['POST'])def predict():  global num_works  if num_works >= max_works:      # 이미 꽉 찼으면    return {"error": "server busy"}, 503  # 요청 거절 (과부하 방지)  num_works += 1       # 작업 시작 -> 카운트 증가  try:    res = model.predict([request.json['data']])    return {"result": res.tolist()}  finally:    num_works -= 1       # 끝나면(성공/실패 상관없이) 카운트 감소

이 방식 장점: 구현 간단, 가벼운 모델에서 서버 다운 방지 / 단점: 예약이 안 되고, 오래 걸리는 작업엔 응답이 지연되며 대규모 서비스엔 부적합

### 3-2. 작업 큐를 이용한 비동기 처리 — 대규모 서비스에서 쓰는 방식 (개념 정리)

메인 프로세스는 요청을 큐(FIFO)에 넣고 즉시 응답만 반환(절대 죽으면 안 됨), 실제 작업은 별도의 Worker 프로세스가 큐에서 꺼내서 처리(Worker가 죽어도 메인 서비스는 안 죽음) — 프린트 스풀러/OS 메시지 큐랑 같은 구조